# Sanskrit Corpus Pairwise Starter

Demonstrates the corpus-level pairwise workflow: embed every file in two corpus folders once, then compare all pairs.

**Workflow:**
1. Point `dir_a` and `dir_b` at folders of `.txt` Sanskrit files
2. Run `sdk.bidirectional_corpus_pairwise(...)` to get A→B and B→A comparisons
3. Browse the synthesis report HTML in your browser

**Equivalent CLI command:**
```bash
python scripts/run_bidirectional_corpus_pairwise.py \
    --dir-a data/corpus_a/ \
    --dir-b data/corpus_b/ \
    --output-dir output/my_run/ \
    --device auto
```

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sanskrit_pipeline.sdk import SanskritResearchSDK
from sanskrit_pipeline.embeddings import DEFAULT_MODEL_ID

In [ ]:
# --- Configure paths ---
DIR_A = Path("../data/corpus_a")   # folder of .txt files for Corpus A
DIR_B = Path("../data/corpus_b")   # folder of .txt files for Corpus B
OUTPUT_DIR = Path("../output/my_bidirectional_run")

LABEL_A = "Corpus A"
LABEL_B = "Corpus B"

# Smoke run: set limits to test with a few files first
LIMIT_A = 2  # set to None for all files
LIMIT_B = 2

print(f"Corpus A: {DIR_A} (limit: {LIMIT_A})")
print(f"Corpus B: {DIR_B} (limit: {LIMIT_B})")
print(f"Output:   {OUTPUT_DIR}")

In [ ]:
sdk = SanskritResearchSDK(
    engine="dandas",
    source_format="devanagari",
    model_id=DEFAULT_MODEL_ID,
    device="auto",
    batch_size=8,
    embedding_progress="batch",
)

In [ ]:
# Run bidirectional corpus pairwise analysis
# This embeds every file once and reuses embeddings across all pairs.
artifacts = sdk.bidirectional_corpus_pairwise(
    dir_a=DIR_A,
    dir_b=DIR_B,
    output_dir=OUTPUT_DIR,
    label_a=LABEL_A,
    label_b=LABEL_B,
    top_k=100,
    limit_a=LIMIT_A,
    limit_b=LIMIT_B,
    generate_reports=True,
)

print(f"Forward manifest:  {artifacts.forward['manifest_json']}")
print(f"Reverse manifest:  {artifacts.reverse['manifest_json']}")
print(f"Synthesis report:  {artifacts.synthesis_report_html}")
print(f"Synthesis CSV:     {artifacts.synthesis_csv}")

In [ ]:
import pandas as pd

# Load the synthesis CSV to inspect ranked pairs
synthesis_df = pd.read_csv(artifacts.synthesis_csv)
print(f"Total pairs: {len(synthesis_df)}")
synthesis_df[["corpus_a_label", "corpus_b_label", "overall_survival", "broad_affinity_survival", "signal_label", "reading_reason"]].head(10)

In [ ]:
# Open the synthesis report in your browser
import webbrowser
webbrowser.open(str(artifacts.synthesis_report_html.resolve()))
print(f"Opened: {artifacts.synthesis_report_html}")